# CV06 — Thanh Phương: Tổng hợp Pipeline & Xuất Báo cáo Tổng kết

**Môn học:** Quản lý & Khai phá Chuỗi Thời gian (QLCTG)  
**Nhóm:** Nhóm 7  
**Thành viên phụ trách:** Thanh Phương  

---

## Tổng quan nhiệm vụ CV06

| Bước | Mô tả |
|:-----|:------|
| **Bước 1** | Kiểm toán đầu ra của toàn nhóm (7/7 bảng, 7/7 biểu đồ) |
| **Bước 2** | Hợp nhất bảng so sánh tổng hợp `final_model_comparison.csv` |
| **Bước 3** | Vẽ Summary Dashboard 4 panel |
| **Bước 4** | Xuất Báo cáo Markdown `docs/BAO_CAO_LAB3_NHOM7.md` |

---

**Đầu vào:**
- `outputs/tables/cv01_data_summary.csv`
- `outputs/tables/cv02_train_history.csv`
- `outputs/tables/cv03_evaluation_metrics.csv`
- `outputs/tables/cv03_feature_importance.csv`
- `outputs/tables/cv04_multistep_metrics.csv`
- `outputs/tables/cv05_best_hyperparameters.csv`
- `outputs/tables/cv05_benchmark_summary.csv`
- `outputs/figures/cv01_raw_and_lags.png` ... `cv05_benchmark_comparison.png`

**Đầu ra:**
- `outputs/tables/final_model_comparison.csv`
- `outputs/figures/cv06_summary_dashboard.png`
- `docs/BAO_CAO_LAB3_NHOM7.md`

## 0. Import thư viện & Cấu hình đường dẫn

In [ ]:
import sys
from pathlib import Path
from typing import Dict, List

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from IPython.display import Image, Markdown, display
import numpy as np
import pandas as pd

# Đường dẫn gốc dự án
PROJECT_ROOT = Path().resolve().parent
FIGURES_DIR  = PROJECT_ROOT / 'outputs' / 'figures'
TABLES_DIR   = PROJECT_ROOT / 'outputs' / 'tables'
DOCS_DIR     = PROJECT_ROOT / 'docs'
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Tables dir   : {TABLES_DIR}')
print(f'Figures dir  : {FIGURES_DIR}')

## Bước 1 — Kiểm toán tiến độ hoàn thành của toàn nhóm

In [ ]:
EXPECTED_TABLES = [
    'cv01_data_summary.csv',
    'cv02_train_history.csv',
    'cv03_evaluation_metrics.csv',
    'cv03_feature_importance.csv',
    'cv04_multistep_metrics.csv',
    'cv05_best_hyperparameters.csv',
    'cv05_benchmark_summary.csv'
]

EXPECTED_FIGURES = [
    'cv01_raw_and_lags.png',
    'cv02_learning_curve.png',
    'cv03_actual_vs_predicted.png',
    'cv03_feature_importance.png',
    'cv04_rolling_features_impact.png',
    'cv04_multistep_forecast.png',
    'cv05_benchmark_comparison.png'
]

print('=' * 60)
print('KIỂM TOÁN DELIVERABLES NHÓM 7')
print('=' * 60)

all_ok = True
print('\n📊 BẢNG SỐ LIỆU:')
for t in EXPECTED_TABLES:
    exists = (TABLES_DIR / t).exists()
    status = '✅' if exists else '❌ THIẾU'
    size   = f"{(TABLES_DIR / t).stat().st_size / 1024:.1f} KB" if exists else '—'
    print(f'  {status}  {t:<45} {size}')
    if not exists:
        all_ok = False

print('\n🖼️  BIỂU ĐỒ TRỰC QUAN:')
for f in EXPECTED_FIGURES:
    exists = (FIGURES_DIR / f).exists()
    status = '✅' if exists else '❌ THIẾU'
    size   = f"{(FIGURES_DIR / f).stat().st_size / 1024:.1f} KB" if exists else '—'
    print(f'  {status}  {f:<45} {size}')
    if not exists:
        all_ok = False

print('\n' + '=' * 60)
if all_ok:
    print('🎉 TẤT CẢ DELIVERABLES ĐÃ ĐẦY ĐỦ — SẴN SÀNG TỔNG HỢP!')
else:
    print('⚠️  Một số deliverables bị thiếu. Cần chạy lại các CV tương ứng.')
print('=' * 60)

## Bước 2 — Hợp nhất bảng so sánh tổng hợp

In [ ]:
rows = []

# CV03 — XGBoost Baseline
cv03_path = TABLES_DIR / 'cv03_evaluation_metrics.csv'
if cv03_path.exists():
    m = pd.read_csv(cv03_path).set_index('metric')['value']
    rows.append({
        'Model': 'XGBoost Baseline (CV02/CV03)',
        'MAE (MW)': round(float(m.get('MAE', float('nan'))), 2),
        'RMSE (MW)': round(float(m.get('RMSE', float('nan'))), 2),
        'MAPE (%)': round(float(m.get('MAPE', float('nan'))), 4),
        'R2 Score': round(float(m.get('R2', float('nan'))), 4),
        'Train Time (s)': '—',
        'Inference (ms)': '—',
        'Ghi chú': 'Default hyperparams, 6 lags'
    })

# CV05 — XGBoost Tuned & Random Forest
cv05b_path = TABLES_DIR / 'cv05_benchmark_summary.csv'
if cv05b_path.exists():
    df5 = pd.read_csv(cv05b_path)
    for _, row in df5.iterrows():
        rows.append({
            'Model': str(row.get('Model', '—')),
            'MAE (MW)': round(float(row.get('MAE (MW)', float('nan'))), 2),
            'RMSE (MW)': round(float(row.get('RMSE (MW)', float('nan'))), 2),
            'MAPE (%)': round(float(row.get('MAPE (%)', float('nan'))), 4),
            'R2 Score': round(float(row.get('R2 Score', float('nan'))), 4),
            'Train Time (s)': round(float(row.get('Train Time (s)', float('nan'))), 2),
            'Inference (ms)': round(float(row.get('Inference (ms)', float('nan'))), 2),
            'Ghi chú': 'TimeSeriesSplit 5-folds GridSearchCV'
        })

# CV04 — Recursive multi-step
cv04s_path = TABLES_DIR / 'cv04_strategy_summary.csv'
if cv04s_path.exists():
    df4s = pd.read_csv(cv04s_path).set_index('strategy')
    if 'Recursive' in df4s.index:
        r = df4s.loc['Recursive']
        rows.append({
            'Model': 'XGBoost Recursive 24h (CV04)',
            'MAE (MW)': round(float(r['MAE']), 2),
            'RMSE (MW)': round(float(r['RMSE']), 2),
            'MAPE (%)': round(float(r['MAPE_pct']), 4),
            'R2 Score': round(float(r['R2']), 4),
            'Train Time (s)': '—',
            'Inference (ms)': '—',
            'Ghi chú': 'Recursive multi-step 24h, rolling features'
        })

df_final = pd.DataFrame(rows)

# Lưu ra file
out_path = TABLES_DIR / 'final_model_comparison.csv'
df_final.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'✅ Đã lưu: {out_path}')
print()
df_final

## Bước 3 — Vẽ Summary Dashboard (4 panels)

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    'Lab 2 — Nhóm 7: Bảng điều khiển tổng hợp kết quả các mô hình',
    fontsize=15, fontweight='bold', y=0.98
)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

models = df_final['Model'].tolist()
short_names = [m.split('(')[0].strip() for m in models]

# Panel 1: MAE
ax1 = fig.add_subplot(gs[0, 0])
maes = df_final['MAE (MW)'].tolist()
bars1 = ax1.bar(range(len(models)), maes, color=colors[:len(models)], edgecolor='white')
ax1.set_xticks(range(len(models)))
ax1.set_xticklabels(short_names, rotation=15, ha='right', fontsize=8)
ax1.set_ylabel('MAE (MW)')
ax1.set_title('MAE so sánh các mô hình', fontweight='bold')
for bar, val in zip(bars1, maes):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
             f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Panel 2: RMSE
ax2 = fig.add_subplot(gs[0, 1])
rmses = df_final['RMSE (MW)'].tolist()
bars2 = ax2.bar(range(len(models)), rmses, color=colors[:len(models)], edgecolor='white')
ax2.set_xticks(range(len(models)))
ax2.set_xticklabels(short_names, rotation=15, ha='right', fontsize=8)
ax2.set_ylabel('RMSE (MW)')
ax2.set_title('RMSE so sánh các mô hình', fontweight='bold')
for bar, val in zip(bars2, rmses):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
             f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Panel 3: Feature Importance top 8
ax3 = fig.add_subplot(gs[1, 0])
df_fi = pd.read_csv(TABLES_DIR / 'cv03_feature_importance.csv').head(8)
bar_colors = ['#2196F3' if t == 'lag' else '#FF9800' for t in df_fi['feature_type']]
ax3.barh(df_fi['feature'][::-1], df_fi['importance'][::-1], color=bar_colors[::-1])
ax3.set_xlabel('Tầm quan trọng (XGBoost gain)')
ax3.set_title('Top 8 Feature Importance (CV03)', fontweight='bold')
ax3.grid(axis='x', alpha=0.3)
legend_elements = [mpatches.Patch(facecolor='#2196F3', label='Lag feature'),
                   mpatches.Patch(facecolor='#FF9800', label='Calendar feature')]
ax3.legend(handles=legend_elements, fontsize=8)

# Panel 4: MAE per step 24h
ax4 = fig.add_subplot(gs[1, 1])
df_ms = pd.read_csv(TABLES_DIR / 'cv04_multistep_metrics.csv')
ax4.plot(df_ms['horizon'], df_ms['MAE'], marker='o', markersize=4,
         color='#E91E63', linewidth=2, label='MAE per step')
ax4.fill_between(df_ms['horizon'], df_ms['MAE'], alpha=0.2, color='#E91E63')
ax4.set_xlabel('Bước dự báo h (giờ)')
ax4.set_ylabel('MAE (MW)')
ax4.set_title('MAE theo bước dự báo Recursive 24h (CV04)', fontweight='bold')
ax4.axvline(x=12, color='gray', linestyle='--', alpha=0.5, label='h=12')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

# Lưu và hiển thị
dashboard_path = FIGURES_DIR / 'cv06_summary_dashboard.png'
plt.savefig(dashboard_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'✅ Đã lưu dashboard: {dashboard_path}')

# Hiển thị trong notebook
display(Image(str(dashboard_path)))

## Bước 4 — Xem báo cáo Markdown đầy đủ

> Chạy cell dưới để tạo lại báo cáo qua script chính, hoặc mở trực tiếp file `docs/BAO_CAO_LAB3_NHOM7.md`.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'src' / 'cv06_report_synthesis.py')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[STDERR]', result.stderr)

In [ ]:
# Hiển thị nội dung báo cáo Markdown trực tiếp trong notebook
report_path = DOCS_DIR / 'BAO_CAO_LAB3_NHOM7.md'
if report_path.exists():
    content = report_path.read_text(encoding='utf-8')
    display(Markdown(content))
else:
    print('Chưa tìm thấy báo cáo. Chạy cell Bước 4 phía trên trước.')

## Phần câu hỏi lý luận — Tóm tắt trực quan

In [ ]:
# In bảng so sánh 3 câu hỏi lý luận cô đọng
ly_luan = pd.DataFrame([
    {
        'Câu hỏi': 'Câu 1: XGBoost vs Linear Regression',
        'Kết luận cốt lõi': 'XGBoost xử lý phi tuyến, tương tác bậc cao, đa cộng tuyến — LR không thể'
    },
    {
        'Câu hỏi': 'Câu 2: Tác động khi tăng bậc trễ p',
        'Kết luận cốt lõi': 'Lợi ích: nắm chu kỳ 24h/168h. Rủi ro: overfitting, NaN, chiều cao'
    },
    {
        'Câu hỏi': 'Câu 3: XGBoost vs ARIMA/SARIMA',
        'Kết luận cốt lõi': 'XGBoost vượt trội khi chuỗi phi tuyến, đa mùa vụ, nhiều biến ngoại sinh, big data'
    }
])
ly_luan.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})

## Tóm tắt kết quả toàn bộ pipeline

In [ ]:
print('=' * 70)
print('KẾT QUẢ PIPELINE — NHÓM 7 — LAB 2 QLCTG')
print('=' * 70)
print(df_final[['Model', 'MAE (MW)', 'RMSE (MW)', 'MAPE (%)', 'R2 Score']].to_string(index=False))
print()
print('🏆 Mô hình tốt nhất: XGBoost Tuned (CV05)')
best = df_final[df_final['Model'].str.contains('tuned', case=False, na=False)]
if not best.empty:
    b = best.iloc[0]
    print(f'   MAE  = {b["MAE (MW)"]} MW  |  RMSE = {b["RMSE (MW)"]} MW')
    print(f'   MAPE = {b["MAPE (%)"]} %   |  R²   = {b["R2 Score"]}')
print()
print('📄 Báo cáo đầy đủ: docs/BAO_CAO_LAB3_NHOM7.md')
print('📊 Dashboard     : outputs/figures/cv06_summary_dashboard.png')
print('📋 Bảng so sánh  : outputs/tables/final_model_comparison.csv')
print('=' * 70)